Paso 1. Conectar con nuestra base vectorial

In [1]:
# Importamos las herramientas necesarias
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

# Utilizamos el mismo modelo de embeddings
# que usamos para crear nuestra base vectorial
modelo_embeddings = OllamaEmbeddings(
    model="bge-m3:latest"
)

# Nos conectamos a la base vectorial existente
base_vectorial = Chroma(
    collection_name="cannabis_derechos_bge_m3",
    embedding_function=modelo_embeddings,
    persist_directory="./chroma_db"
)

# Verificamos que los documentos estén disponibles
total_fragmentos = base_vectorial._collection.count()

print("Fragmentos disponibles:", total_fragmentos)

Fragmentos disponibles: 778


Paso 2. Crear las preguntas de evaluación

In [2]:
# ==================================================
# NOTEBOOK 04 - EVALUACIÓN DE LA RECUPERACIÓN
# ==================================================

# Creamos preguntas y definimos qué documento
# esperamos que ChromaDB recupere en cada caso.

casos_evaluacion = [

    # DERECHOS DE LOS PACIENTES

    {
        "pregunta": "¿Qué derechos tiene un paciente sobre su historia clínica?",
        "documento_esperado": "ley_26529_derechos_paciente.pdf"
    },

    {
        "pregunta": "¿Quién es el titular de la historia clínica?",
        "documento_esperado": "ley_26529_derechos_paciente.pdf"
    },

    {
        "pregunta": "¿Qué es el consentimiento informado?",
        "documento_esperado": "ley_26529_derechos_paciente.pdf"
    },

    # CANNABIS MEDICINAL Y REPROCANN

    {
        "pregunta": "¿Cuál es el objetivo de la Ley 27.350?",
        "documento_esperado": "TEXTO ACTUALIZADO - Ley 27350 - INVESTIGACION MEDICA Y CIENTIFICA _ Argentina.gob.ar.pdf"
    },

    {
        "pregunta": "¿Qué establece la Resolución 1780/2025 sobre REPROCANN?",
        "documento_esperado": "BOLETIN OFICIAL REPUBLICA ARGENTINA - MINISTERIO DE SALUD - Resolución 1780_2025.pdf"
    },

    {
        "pregunta": "¿Qué requisitos establece la Resolución 800/2021 para REPROCANN?",
        "documento_esperado": "TEXTO ACTUALIZADO - Resolución 800_2021 - MINISTERIO DE SALUD _ Argentina.gob.ar.pdf"
    },

    # PROTECCIÓN DE DATOS PERSONALES

    {
        "pregunta": "¿Qué establece la Ley 25.326 sobre los datos personales?",
        "documento_esperado": "PROTECCION DE LOS DATOS.pdf"
    },

    # DERECHOS Y PROCEDIMIENTOS POLICIALES

    {
        "pregunta": "¿Qué derechos tiene una persona durante un allanamiento?",
        "documento_esperado": "ATAJO_Dip Allanamientos-digital-a 150.pdf"
    },

    {
        "pregunta": "¿Qué derechos tiene una persona imputada en un proceso penal?",
        "documento_esperado": "Derechos de la persona imputada _ Unidad Especial Estratégica para la Implementación del Sistema Procesal Penal Acusatorio.pdf"
    }
]

# Verificamos cuántas preguntas tenemos
print("Cantidad de preguntas de evaluación:", len(casos_evaluacion))

# Mostramos las preguntas
for numero, caso in enumerate(casos_evaluacion, start=1):
    print(f"{numero}. {caso['pregunta']}")

Cantidad de preguntas de evaluación: 9
1. ¿Qué derechos tiene un paciente sobre su historia clínica?
2. ¿Quién es el titular de la historia clínica?
3. ¿Qué es el consentimiento informado?
4. ¿Cuál es el objetivo de la Ley 27.350?
5. ¿Qué establece la Resolución 1780/2025 sobre REPROCANN?
6. ¿Qué requisitos establece la Resolución 800/2021 para REPROCANN?
7. ¿Qué establece la Ley 25.326 sobre los datos personales?
8. ¿Qué derechos tiene una persona durante un allanamiento?
9. ¿Qué derechos tiene una persona imputada en un proceso penal?


Paso 3. Evaluar la recuperación de documentos

In [3]:
# ==================================================
# PASO 3 - EVALUAR LA RECUPERACIÓN DE DOCUMENTOS
# ==================================================

resultados_evaluacion = []

for numero, caso in enumerate(casos_evaluacion, start=1):

    pregunta = caso["pregunta"]
    documento_esperado = caso["documento_esperado"]

    # Recuperamos los 3 fragmentos más relevantes
    documentos = base_vectorial.similarity_search(
        pregunta,
        k=3
    )

    # Obtenemos los nombres de los documentos recuperados
    fuentes_recuperadas = [
        doc.metadata.get("fuente", "")
        for doc in documentos
    ]

    # Verificamos si aparece el documento esperado
    encontrado = documento_esperado in fuentes_recuperadas

    # Guardamos los resultados para analizarlos después
    resultados_evaluacion.append({
        "pregunta": pregunta,
        "documento_esperado": documento_esperado,
        "fuentes_recuperadas": fuentes_recuperadas,
        "encontrado": encontrado
    })

    # Mostramos el resultado de cada prueba
    print(f"\nPRUEBA {numero}")
    print("Pregunta:", pregunta)
    print("Documento esperado:", documento_esperado)
    print("¿Fue recuperado?:", "SÍ ✅" if encontrado else "NO ❌")

# Calculamos el porcentaje de preguntas
# donde se recuperó el documento esperado

aciertos = sum(
    resultado["encontrado"]
    for resultado in resultados_evaluacion
)

porcentaje = (aciertos / len(resultados_evaluacion)) * 100

print("\n" + "=" * 50)
print("RESULTADO GENERAL")
print(f"Documentos encontrados: {aciertos} de {len(resultados_evaluacion)}")
print(f"Porcentaje de aciertos: {porcentaje:.2f}%")


PRUEBA 1
Pregunta: ¿Qué derechos tiene un paciente sobre su historia clínica?
Documento esperado: ley_26529_derechos_paciente.pdf
¿Fue recuperado?: SÍ ✅

PRUEBA 2
Pregunta: ¿Quién es el titular de la historia clínica?
Documento esperado: ley_26529_derechos_paciente.pdf
¿Fue recuperado?: SÍ ✅

PRUEBA 3
Pregunta: ¿Qué es el consentimiento informado?
Documento esperado: ley_26529_derechos_paciente.pdf
¿Fue recuperado?: SÍ ✅

PRUEBA 4
Pregunta: ¿Cuál es el objetivo de la Ley 27.350?
Documento esperado: TEXTO ACTUALIZADO - Ley 27350 - INVESTIGACION MEDICA Y CIENTIFICA _ Argentina.gob.ar.pdf
¿Fue recuperado?: SÍ ✅

PRUEBA 5
Pregunta: ¿Qué establece la Resolución 1780/2025 sobre REPROCANN?
Documento esperado: BOLETIN OFICIAL REPUBLICA ARGENTINA - MINISTERIO DE SALUD - Resolución 1780_2025.pdf
¿Fue recuperado?: SÍ ✅

PRUEBA 6
Pregunta: ¿Qué requisitos establece la Resolución 800/2021 para REPROCANN?
Documento esperado: TEXTO ACTUALIZADO - Resolución 800_2021 - MINISTERIO DE SALUD _ Argentina.

In [4]:
# Analizamos la pregunta que no recuperó
# el documento esperado entre los primeros 3 resultados.

pregunta = (
    "¿Qué requisitos establece la Resolución 800/2021 "
    "para REPROCANN?"
)

# Recuperamos 10 fragmentos para comparar
documentos = base_vectorial.similarity_search(
    pregunta,
    k=10
)

print("RESULTADOS DE LA BÚSQUEDA:\n")

for numero, doc in enumerate(documentos, start=1):

    print(f"RESULTADO {numero}")
    print("Documento:", doc.metadata.get("fuente"))
    print("Página:", doc.metadata.get("pagina"))
    print("-" * 50)

RESULTADOS DE LA BÚSQUEDA:

RESULTADO 1
Documento: BOLETIN OFICIAL REPUBLICA ARGENTINA - MINISTERIO DE SALUD - Resolución 1780_2025.pdf
Página: 8
--------------------------------------------------
RESULTADO 2
Documento: BOLETIN OFICIAL REPUBLICA ARGENTINA - MINISTERIO DE SALUD - Resolución 1780_2025.pdf
Página: 9
--------------------------------------------------
RESULTADO 3
Documento: BOLETIN OFICIAL REPUBLICA ARGENTINA - MINISTERIO DE SALUD - Resolución 1780_2025.pdf
Página: 5
--------------------------------------------------
RESULTADO 4
Documento: BOLETIN OFICIAL REPUBLICA ARGENTINA - MINISTERIO DE SALUD - Resolución 1780_2025.pdf
Página: 7
--------------------------------------------------
RESULTADO 5
Documento: BOLETIN OFICIAL REPUBLICA ARGENTINA - MINISTERIO DE SALUD - Resolución 1780_2025.pdf
Página: 7
--------------------------------------------------
RESULTADO 6
Documento: TEXTO ACTUALIZADO - Resolución 800_2021 - MINISTERIO DE SALUD _ Argentina.gob.ar.pdf
Página: 8
---------

In [5]:
# Comparamos la recuperación usando diferentes valores de k

for k in [3, 5, 10]:

    aciertos = 0

    for caso in casos_evaluacion:

        documentos = base_vectorial.similarity_search(
            caso["pregunta"],
            k=k
        )

        fuentes = [
            doc.metadata.get("fuente", "")
            for doc in documentos
        ]

        if caso["documento_esperado"] in fuentes:
            aciertos += 1

    porcentaje = aciertos / len(casos_evaluacion) * 100

    print(
        f"Hit Rate@{k}: "
        f"{aciertos}/{len(casos_evaluacion)} "
        f"({porcentaje:.2f}%)"
    )

Hit Rate@3: 8/9 (88.89%)
Hit Rate@5: 8/9 (88.89%)
Hit Rate@10: 9/9 (100.00%)


In [6]:
# ==================================================
# PASO 6 - CALCULAR MRR
# ==================================================

rangos_reciprocos = []

for numero, caso in enumerate(casos_evaluacion, start=1):

    documentos = base_vectorial.similarity_search(
        caso["pregunta"],
        k=10
    )

    fuentes = [
        doc.metadata.get("fuente", "")
        for doc in documentos
    ]

    # Buscamos la primera posición del documento esperado
    if caso["documento_esperado"] in fuentes:
        posicion = fuentes.index(caso["documento_esperado"]) + 1
        rango_reciproco = 1 / posicion
    else:
        posicion = None
        rango_reciproco = 0

    rangos_reciprocos.append(rango_reciproco)

    print(f"PRUEBA {numero}")
    print("Primera posición:", posicion)
    print(f"Rango recíproco: {rango_reciproco:.3f}")
    print("-" * 40)

# Calculamos el promedio
mrr = sum(rangos_reciprocos) / len(rangos_reciprocos)

print("\nRESULTADO GENERAL")
print(f"MRR@10: {mrr:.3f}")

PRUEBA 1
Primera posición: 1
Rango recíproco: 1.000
----------------------------------------
PRUEBA 2
Primera posición: 1
Rango recíproco: 1.000
----------------------------------------
PRUEBA 3
Primera posición: 1
Rango recíproco: 1.000
----------------------------------------
PRUEBA 4
Primera posición: 2
Rango recíproco: 0.500
----------------------------------------
PRUEBA 5
Primera posición: 2
Rango recíproco: 0.500
----------------------------------------
PRUEBA 6
Primera posición: 6
Rango recíproco: 0.167
----------------------------------------
PRUEBA 7
Primera posición: 1
Rango recíproco: 1.000
----------------------------------------
PRUEBA 8
Primera posición: 1
Rango recíproco: 1.000
----------------------------------------
PRUEBA 9
Primera posición: 1
Rango recíproco: 1.000
----------------------------------------

RESULTADO GENERAL
MRR@10: 0.796


In [7]:
# ==================================================
# PASO 7 - PRECISION@3
# ==================================================

valores_precision = []

for numero, caso in enumerate(casos_evaluacion, start=1):

    documentos = base_vectorial.similarity_search(
        caso["pregunta"],
        k=3
    )

    # Contamos cuántos fragmentos pertenecen
    # al documento esperado
    relevantes = sum(
        doc.metadata.get("fuente") == caso["documento_esperado"]
        for doc in documentos
    )

    precision = relevantes / len(documentos)

    valores_precision.append(precision)

    print(
        f"PRUEBA {numero}: "
        f"{relevantes} de {len(documentos)} fragmentos "
        f"del documento esperado "
        f"- Precision@3: {precision:.3f}"
    )

# Promedio de las nueve preguntas
precision_promedio = sum(valores_precision) / len(valores_precision)

print("\n" + "=" * 45)
print("RESULTADO GENERAL")
print(f"Precision@3 (aproximada): {precision_promedio:.3f}")
print(f"Porcentaje: {precision_promedio * 100:.2f}%")

PRUEBA 1: 3 de 3 fragmentos del documento esperado - Precision@3: 1.000
PRUEBA 2: 3 de 3 fragmentos del documento esperado - Precision@3: 1.000
PRUEBA 3: 2 de 3 fragmentos del documento esperado - Precision@3: 0.667
PRUEBA 4: 1 de 3 fragmentos del documento esperado - Precision@3: 0.333
PRUEBA 5: 1 de 3 fragmentos del documento esperado - Precision@3: 0.333
PRUEBA 6: 0 de 3 fragmentos del documento esperado - Precision@3: 0.000
PRUEBA 7: 3 de 3 fragmentos del documento esperado - Precision@3: 1.000
PRUEBA 8: 3 de 3 fragmentos del documento esperado - Precision@3: 1.000
PRUEBA 9: 2 de 3 fragmentos del documento esperado - Precision@3: 0.667

RESULTADO GENERAL
Precision@3 (aproximada): 0.667
Porcentaje: 66.67%


### Evaluación de la recuperación de información

Se evaluó el sistema RAG mediante nueve preguntas relacionadas con cannabis medicinal, REPROCANN, derechos de los pacientes, protección de datos personales y procedimientos policiales en Argentina.

Para cada pregunta se identificó previamente un documento esperado y se analizaron los resultados obtenidos mediante búsqueda semántica en ChromaDB, utilizando el modelo de embeddings `bge-m3`.

Los resultados fueron los siguientes:

- **Hit Rate@3:** 88,89 %.
- **Hit Rate@5:** 88,89 %.
- **Hit Rate@10:** 100 %.
- **MRR@10:** 0,796.
- **Precision@3 aproximada:** 66,67 %.

Los resultados muestran que el sistema logra recuperar los documentos esperados en la mayoría de las consultas y que, generalmente, los posiciona entre los primeros resultados.

Se identificó una dificultad particular en la consulta sobre la Resolución 800/2021, ya que el sistema priorizó fragmentos de la Resolución 1780/2025. El documento esperado apareció recién en la sexta posición.

Esta situación evidencia una limitación de la búsqueda semántica cuando existen documentos normativos relacionados entre sí.

Como posible mejora, se propone incorporar mecanismos de recuperación híbrida, filtros por normativa y estrategias de reordenamiento de resultados (*reranking*).

La evaluación constituye una primera aproximación al rendimiento del recuperador. Para obtener una medición más completa sería necesario ampliar el conjunto de preguntas y evaluar manualmente la relevancia de cada fragmento recuperado.